# External scores

Everything derived from the purchased `EXT_SOURCE` scores, isolated here so the bought-vs-built split is airtight. Raw scores kept as `EXT_SOURCE_*`; every engineered feature prefixed `ext_calc_`. Saved to `ext.pkl` and merged by `load_master`.

In [4]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.data import save_features

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
cols = ["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH", "DAYS_EMPLOYED"]
df = pd.concat([pd.read_csv(RAW / "application_train.csv", usecols=cols),
                pd.read_csv(RAW / "application_test.csv", usecols=cols)], ignore_index=True)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
df.shape

(356255, 6)

## Features

In [5]:
out = df[["SK_ID_CURR"] + ext].copy()
out["ext_calc_mean"] = df[ext].mean(axis=1)
out["ext_calc_min"] = df[ext].min(axis=1)
out["ext_calc_max"] = df[ext].max(axis=1)
out["ext_calc_median"] = df[ext].median(axis=1)
out["ext_calc_weighted"] = df["EXT_SOURCE_1"] * 2 + df["EXT_SOURCE_2"] * 3 + df["EXT_SOURCE_3"] * 4
out["ext_calc_std"] = df[ext].std(axis=1)
out["ext_calc_nan_count"] = df[ext].isna().sum(axis=1)
out["ext_calc_prod"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
out["ext_calc_1x2"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"]
out["ext_calc_1x3"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_3"]
out["ext_calc_2x3"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
out["ext_calc_mean_to_birth"] = out["ext_calc_mean"] / df["DAYS_BIRTH"]
out["ext_calc_mean_to_employ"] = out["ext_calc_mean"] / df["DAYS_EMPLOYED"]
for i, c in enumerate(ext, 1):
    out[f"ext_calc_{i}_to_birth"] = df[c] / df["DAYS_BIRTH"]
    out[f"ext_calc_{i}_to_employ"] = df[c] / df["DAYS_EMPLOYED"]
out = out.replace([np.inf, -np.inf], np.nan)
out.shape

(356255, 23)

# Save

In [6]:
save_features(out, "ext"); out.shape

(356255, 23)